# CIFAR-10 CNN: HOOI Tucker-2

`00_fundamentals/02_hooi.ipynb` で実装したHOOIの考え方を、学習済みCNNの `conv2` に対する partial Tucker-2 へ適用する。

03までの **baseline / data split / 02で自動選択したbalanced rank** を引き継ぎ、同じrank・同じTucker-2構造で次を比較する。

- 自作HOSVD
- 自作HOOI
- TensorLy `partial_tucker(init="svd")`

ここではfine-tuningを行わず、**分解法そのものの違い**を見る。

> **自分で実装する重要部分**
>
> `tucker2_hooi_sweep()` の `U_out` / `U_in` 更新だけ。
>
> データ準備、比較、計測、TensorLy検証、任意の推論benchmarkは完成形にしてある。


In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import copy
import sys

from IPython.display import display
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import tensorly as tl
from tensorly.decomposition import partial_tucker

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (p / "src/nn_compression").is_dir():
        sys.path.insert(0, str(p / "src"))
        break

from nn_compression.compression import hosvd, reconstruct_tucker, truncated_svd
from nn_compression.datasets import shuffled_index_splits
from nn_compression.metrics import (
    count_parameters,
    parameters_reduction,
    relative_frobenius_error,
)
from nn_compression.models import CIFAR10CNN
from nn_compression.tensor import mode_dot, unfold
from nn_compression.training import evaluate
from nn_compression.utils import find_project_root, get_experiment_dirs, set_seed

tl.set_backend("pytorch")

root = find_project_root(Path.cwd())
_, _, results_dir = get_experiment_dirs(
    root,
    "20_tucker",
    "10_cifar10_cnn",
    "04_hooi_tucker2",
)

selected_csv = (
    root
    / "results/20_tucker/10_cifar10_cnn/02_rank_sweep/selected_rank_settings.csv"
)
ft_csv = (
    root
    / "results/20_tucker/10_cifar10_cnn/03_finetuning/finetuning_comparison.csv"
)
model_path = (
    root
    / "models/10_svd/40_cifar10_cnn/"
      "02_svd_global_compression_using_src_corrected/cifar10_cnn_baseline.pt"
)

SEED = 0
BATCH_SIZE = 256

set_seed(SEED)
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)


## 1. 03と同じ評価条件を準備

rankは02の `selected_rank_settings.csv` から **balanced** を自動取得する。rank searchは04では行わない。

HOOIの反復停止はweight reconstruction errorだけで決め、validation / testはrank選択や収束判定に使わない。


In [ ]:
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

full_train = datasets.CIFAR10(
    root=root / "data",
    train=True,
    download=False,
    transform=evaluation_transform,
)
test_dataset = datasets.CIFAR10(
    root=root / "data",
    train=False,
    download=False,
    transform=evaluation_transform,
)

_, _, val_rank_idx = shuffled_index_splits(
    len(full_train),
    (40_000, 5_000, 5_000),
    seed=SEED,
)

val_rank = Subset(full_train, val_rank_idx)
val_loader = DataLoader(
    val_rank,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

if not selected_csv.is_file():
    raise FileNotFoundError(
        f"{selected_csv}\n先に 02_rank_sweep.ipynb を実行してください。"
    )

selected = pd.read_csv(selected_csv)
balanced_rows = selected[selected["role"] == "balanced"]

if len(balanced_rows) != 1:
    raise ValueError("selected_rank_settings.csv の balanced は1行である必要があります。")

balanced_row = balanced_rows.iloc[0]
rank_out = int(balanced_row["rank_out"])
rank_in = int(balanced_row["rank_in"])

baseline = CIFAR10CNN().to(device)
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
state_dict = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint
    else checkpoint
)
baseline.load_state_dict(state_dict)
baseline.eval()

criterion = nn.CrossEntropyLoss()

base_val_loss, base_val_acc = evaluate(
    baseline, val_loader, criterion, device
)
base_test_loss, base_test_acc = evaluate(
    baseline, test_loader, criterion, device
)

print("balanced rank:", (rank_out, rank_in))
print(f"baseline validation accuracy: {base_val_acc:.4f}")
print(f"baseline test accuracy      : {base_test_acc:.4f}")


## 2. 分解時間を公平に測るための補助関数

CUDA / MPSでは処理が非同期に実行されることがあるため、計測の前後でdeviceを同期する。

ここで記録する `decomposition_time_ms` は、**weightを受け取って分解結果を返すまで**の時間である。モデル評価時間やfine-tuning時間は含めない。


In [ ]:
def synchronize_device() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch.mps, "synchronize"):
        torch.mps.synchronize()


def timed_call(fn):
    synchronize_device()
    start = perf_counter()
    result = fn()
    synchronize_device()
    elapsed_ms = (perf_counter() - start) * 1000.0
    return result, elapsed_ms


## 3. HOSVD Tucker-2を基準にする

`conv2.weight` のshapeは

```text
(C_out, C_in, kH, kW)
```

で、mode 0 / 1だけを `(rank_out, rank_in)` に圧縮する。

比較を分解法だけに限定するため、HOSVD / HOOI / TensorLyのどれも同じ

```text
1×1 Conv: C_in → rank_in
k×k Conv: rank_in → rank_out
1×1 Conv: rank_out → C_out
```

へ組み立てる。


In [ ]:
def build_from_components(
    conv: nn.Conv2d,
    core: torch.Tensor,
    u_out: torch.Tensor,
    u_in: torch.Tensor,
) -> nn.Sequential:
    if conv.groups != 1:
        raise ValueError("このNotebookでは groups=1 のConv2dのみ扱います。")

    factory_kwargs = {
        "device": conv.weight.device,
        "dtype": conv.weight.dtype,
    }
    r_out = u_out.shape[1]
    r_in = u_in.shape[1]

    left = nn.Conv2d(
        conv.in_channels,
        r_in,
        kernel_size=1,
        bias=False,
        **factory_kwargs,
    )
    middle = nn.Conv2d(
        r_in,
        r_out,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        bias=False,
        padding_mode=conv.padding_mode,
        **factory_kwargs,
    )
    right = nn.Conv2d(
        r_out,
        conv.out_channels,
        kernel_size=1,
        bias=(conv.bias is not None),
        **factory_kwargs,
    )

    with torch.no_grad():
        left.weight.copy_(u_in.T[:, :, None, None])
        middle.weight.copy_(core)
        right.weight.copy_(u_out[:, :, None, None])
        if conv.bias is not None:
            right.bias.copy_(conv.bias)

    return nn.Sequential(left, middle, right)


In [ ]:
weight = baseline.conv2.weight.detach()
ranks = {0: rank_out, 1: rank_in}

(core_h, factors_h), hosvd_time_ms = timed_call(
    lambda: hosvd(weight, ranks)
)

hat_h = reconstruct_tucker(core_h, factors_h)
err_h = float(relative_frobenius_error(weight, hat_h))

model_h = copy.deepcopy(baseline)
model_h.conv2 = build_from_components(
    model_h.conv2,
    core_h,
    factors_h[0],
    factors_h[1],
).to(device)

h_val_loss, h_val_acc = evaluate(
    model_h, val_loader, criterion, device
)
h_test_loss, h_test_acc = evaluate(
    model_h, test_loader, criterion, device
)

print("weight shape:", tuple(weight.shape))
print("core shape  :", tuple(core_h.shape))
print(f"HOSVD weight relative error : {err_h:.6f}")
print(f"HOSVD decomposition time    : {hosvd_time_ms:.3f} ms")
print(f"HOSVD validation accuracy   : {h_val_acc:.4f}")
print(f"HOSVD test accuracy         : {h_test_acc:.4f}")


## 4. 【重要】partial HOOIの1 sweepを自分で実装する

HOSVDで得た `U_out`, `U_in` を初期値にする。

### `U_out` の更新

\[
W
\times_1 U_{\mathrm{in}}^\mathsf{T}
\rightarrow
\mathrm{unfold}_0
\rightarrow
\mathrm{SVD}
\rightarrow
U_{\mathrm{out}}
\]

### `U_in` の更新

\[
W
\times_0 U_{\mathrm{out}}^\mathsf{T}
\rightarrow
\mathrm{unfold}_1
\rightarrow
\mathrm{SVD}
\rightarrow
U_{\mathrm{in}}
\]

**mode 1を更新するときは、このsweep内ですでに更新された新しい `U_out` を使う。**

mode 2 / 3 (`kH`, `kW`) は圧縮しない。


In [ ]:
def tucker2_hooi_sweep(
    weight: torch.Tensor,
    factors: dict[int, torch.Tensor],
    rank_out: int,
    rank_in: int,
) -> dict[int, torch.Tensor]:
    updated = {
        mode: factor.clone()
        for mode, factor in factors.items()
    }

    # TODO 1: U_out (mode 0) を更新する
    #
    # ヒント:
    # - weightを現在の U_in.T でmode 1方向へ射影
    # - 射影後Tensorをmode 0でunfold
    # - truncated_svd(..., rank_out) の左特異ベクトルを updated[0] にする

    # TODO 2: U_in (mode 1) を更新する
    #
    # ヒント:
    # - weightを「TODO 1で更新済み」の U_out.T でmode 0方向へ射影
    # - 射影後Tensorをmode 1でunfold
    # - truncated_svd(..., rank_in) の左特異ベクトルを updated[1] にする

    raise NotImplementedError(
        "TODO: partial HOOI の mode 0 / 1 update を実装してください。"
    )

    # return updated


## 5. HOOIを反復する

ここは完成形。

1 sweepごとにcoreを作り直し、元weightを再構成してrelative Frobenius errorを記録する。

前回からの改善量が `tol` 未満になったら終了する。validation accuracyを停止条件に使わない点が重要。


In [ ]:
def core_from_factors(
    weight: torch.Tensor,
    factors: dict[int, torch.Tensor],
) -> torch.Tensor:
    core = mode_dot(weight, factors[0].T, 0)
    core = mode_dot(core, factors[1].T, 1)
    return core


def tucker2_hooi(
    weight: torch.Tensor,
    rank_out: int,
    rank_in: int,
    max_iters: int = 20,
    tol: float = 1e-6,
):
    _, factors = hosvd(
        weight,
        {0: rank_out, 1: rank_in},
    )
    factors = {
        mode: factor.clone()
        for mode, factor in factors.items()
    }

    history = []
    previous_error = None

    for iteration in range(1, max_iters + 1):
        factors = tucker2_hooi_sweep(
            weight,
            factors,
            rank_out,
            rank_in,
        )

        core = core_from_factors(weight, factors)
        reconstructed = reconstruct_tucker(core, factors)
        error = float(
            relative_frobenius_error(weight, reconstructed)
        )

        improvement = (
            None
            if previous_error is None
            else previous_error - error
        )
        history.append({
            "iteration": iteration,
            "weight_relative_error": error,
            "improvement": improvement,
        })

        if (
            improvement is not None
            and 0.0 <= improvement < tol
        ):
            break

        previous_error = error

    core = core_from_factors(weight, factors)
    return core, factors, pd.DataFrame(history)


In [ ]:
(core_o, factors_o, hooi_history), hooi_time_ms = timed_call(
    lambda: tucker2_hooi(
        weight,
        rank_out,
        rank_in,
        max_iters=20,
        tol=1e-6,
    )
)

hat_o = reconstruct_tucker(core_o, factors_o)
err_o = float(relative_frobenius_error(weight, hat_o))

print(f"HOSVD weight relative error : {err_h:.6f}")
print(f"HOOI  weight relative error : {err_o:.6f}")
print(f"HOOI decomposition time     : {hooi_time_ms:.3f} ms")
print(f"HOOI iterations             : {len(hooi_history)}")

display(hooi_history)


## 6. 自作HOOIをCNNへ入れて圧縮直後を評価

fine-tuningはしない。

HOSVDとHOOIのrank・parameters・MACs・層構造を同じにし、factor/coreの求め方だけを変える。


In [ ]:
model_o = copy.deepcopy(baseline)
model_o.conv2 = build_from_components(
    model_o.conv2,
    core_o,
    factors_o[0],
    factors_o[1],
).to(device)

o_val_loss, o_val_acc = evaluate(
    model_o, val_loader, criterion, device
)
o_test_loss, o_test_acc = evaluate(
    model_o, test_loader, criterion, device
)

print(f"HOOI validation accuracy: {o_val_acc:.4f}")
print(f"HOOI test accuracy      : {o_test_acc:.4f}")


## 7. TensorLy `partial_tucker(init="svd")` と比較

TensorLy側も同じmode 0 / 1、同じ `(rank_out, rank_in)` を使う。

TensorLyの `partial_tucker` は反復的なTucker最適化（HOOI）を行うため、自作HOOIの検算相手になる。

factorそのものは符号や回転の自由度があるので直接一致を要求せず、**再構成誤差とCNNの評価値**で比較する。


In [ ]:
def run_tensorly_partial_tucker():
    return partial_tucker(
        weight,
        rank=[rank_out, rank_in],
        modes=[0, 1],
        init="svd",
        n_iter_max=20,
        tol=1e-6,
    )


((core_tl, factors_tl), tl_reported_errors), tensorly_time_ms = timed_call(
    run_tensorly_partial_tucker
)

factors_tl_dict = {
    0: factors_tl[0],
    1: factors_tl[1],
}

hat_tl = reconstruct_tucker(core_tl, factors_tl_dict)
err_tl = float(relative_frobenius_error(weight, hat_tl))

tensorly_history = pd.DataFrame({
    "iteration": range(1, len(tl_reported_errors) + 1),
    "tensorly_reported_error": [
        float(error)
        for error in tl_reported_errors
    ],
})

model_tl = copy.deepcopy(baseline)
model_tl.conv2 = build_from_components(
    model_tl.conv2,
    core_tl,
    factors_tl_dict[0],
    factors_tl_dict[1],
).to(device)

tl_val_loss, tl_val_acc = evaluate(
    model_tl, val_loader, criterion, device
)
tl_test_loss, tl_test_acc = evaluate(
    model_tl, test_loader, criterion, device
)

print(f"TensorLy weight relative error : {err_tl:.6f}")
print(f"TensorLy decomposition time    : {tensorly_time_ms:.3f} ms")
print(f"TensorLy iterations            : {len(tensorly_history)}")
print(f"TensorLy validation accuracy   : {tl_val_acc:.4f}")
print(f"TensorLy test accuracy         : {tl_test_acc:.4f}")

display(tensorly_history)


## 8. HOSVD / 自作HOOI / TensorLyを同じ表で比較

同じbalanced rankなので、3つの圧縮モデルは同じparameters / theoretical MACsになるはず。

`accuracy_drop` は

```text
baseline validation accuracy - 圧縮後 validation accuracy
```

とする。負ならbaselineよりaccuracyが高い。


In [ ]:
compressed_parameter_count = count_parameters(model_h)

assert count_parameters(model_o) == compressed_parameter_count
assert count_parameters(model_tl) == compressed_parameter_count

parameter_reduction = parameters_reduction(
    baseline,
    model_h,
)
conv2_macs_reduction = float(
    balanced_row["conv2_macs_reduction"]
)
all_macs_reduction = float(
    balanced_row["all_macs_reduction"]
)

comparison = pd.DataFrame([
    {
        "method": "baseline",
        "rank_out": None,
        "rank_in": None,
        "weight_relative_error": None,
        "decomposition_time_ms": None,
        "iterations": None,
        "validation_loss": base_val_loss,
        "validation_acc": base_val_acc,
        "accuracy_drop": 0.0,
        "test_loss": base_test_loss,
        "test_acc": base_test_acc,
        "parameters": count_parameters(baseline),
        "parameters_reduction": 0.0,
        "conv2_macs_reduction": None,
        "all_macs_reduction": None,
    },
    {
        "method": "hosvd_tucker2",
        "rank_out": rank_out,
        "rank_in": rank_in,
        "weight_relative_error": err_h,
        "decomposition_time_ms": hosvd_time_ms,
        "iterations": 0,
        "validation_loss": h_val_loss,
        "validation_acc": h_val_acc,
        "accuracy_drop": base_val_acc - h_val_acc,
        "test_loss": h_test_loss,
        "test_acc": h_test_acc,
        "parameters": compressed_parameter_count,
        "parameters_reduction": parameter_reduction,
        "conv2_macs_reduction": conv2_macs_reduction,
        "all_macs_reduction": all_macs_reduction,
    },
    {
        "method": "self_hooi_tucker2",
        "rank_out": rank_out,
        "rank_in": rank_in,
        "weight_relative_error": err_o,
        "decomposition_time_ms": hooi_time_ms,
        "iterations": len(hooi_history),
        "validation_loss": o_val_loss,
        "validation_acc": o_val_acc,
        "accuracy_drop": base_val_acc - o_val_acc,
        "test_loss": o_test_loss,
        "test_acc": o_test_acc,
        "parameters": compressed_parameter_count,
        "parameters_reduction": parameter_reduction,
        "conv2_macs_reduction": conv2_macs_reduction,
        "all_macs_reduction": all_macs_reduction,
    },
    {
        "method": "tensorly_partial_tucker",
        "rank_out": rank_out,
        "rank_in": rank_in,
        "weight_relative_error": err_tl,
        "decomposition_time_ms": tensorly_time_ms,
        "iterations": len(tensorly_history),
        "validation_loss": tl_val_loss,
        "validation_acc": tl_val_acc,
        "accuracy_drop": base_val_acc - tl_val_acc,
        "test_loss": tl_test_loss,
        "test_acc": tl_test_acc,
        "parameters": compressed_parameter_count,
        "parameters_reduction": parameter_reduction,
        "conv2_macs_reduction": conv2_macs_reduction,
        "all_macs_reduction": all_macs_reduction,
    },
])

display(comparison)


## 9. 03 Fine-tuning結果を参考として確認

03はすでにHOSVD Tucker-2をfine-tuningした実験。

04ではfine-tuningせず、まず **圧縮直後のHOSVD vs HOOI** を評価する。03のbalanced結果は参考として別表で確認するだけにする。


In [ ]:
if ft_csv.is_file():
    finetuning_results = pd.read_csv(ft_csv)

    if "role" in finetuning_results.columns:
        display(
            finetuning_results[
                finetuning_results["role"] == "balanced"
            ]
        )
    else:
        display(finetuning_results)
else:
    print("03のfinetuning_comparison.csvがないため、参考表示をskipします。")


## 10. 結果を保存

HOOIの反復履歴とTensorLyの反復履歴は分けて保存する。

TensorLyの履歴列はTensorLy自身が返すreported errorで、最終比較表の `weight_relative_error` は全手法とも自作 `relative_frobenius_error()` で再計算した値。


In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)

comparison_path = results_dir / "hooi_comparison.csv"
hooi_history_path = results_dir / "hooi_error_history.csv"
tensorly_history_path = results_dir / "tensorly_error_history.csv"

comparison.to_csv(comparison_path, index=False)
hooi_history.to_csv(hooi_history_path, index=False)
tensorly_history.to_csv(tensorly_history_path, index=False)

print("saved:", comparison_path)
print("saved:", hooi_history_path)
print("saved:", tensorly_history_path)


## 11. 【任意】実測inference benchmark

MACs削減率と実測速度は別物。

Tucker-2は1つのConv2dを3つのConv2dへ置き換えるため、kernel launchやmemory accessの影響で、MACsが減っても同じ割合で高速になるとは限らない。

必要になったら `RUN_INFERENCE_BENCHMARK = True` にして、同じdevice / batch / warm-up条件で比較する。

> benchmarkは環境依存なので、結果は「このPC・このdeviceでの実測値」として扱う。


In [ ]:
RUN_INFERENCE_BENCHMARK = False
BENCHMARK_WARMUP = 20
BENCHMARK_REPEATS = 100


@torch.inference_mode()
def benchmark_inference(
    model: nn.Module,
    inputs: torch.Tensor,
    warmup: int = 20,
    repeats: int = 100,
) -> float:
    model.eval()

    for _ in range(warmup):
        model(inputs)

    synchronize_device()
    start = perf_counter()

    for _ in range(repeats):
        model(inputs)

    synchronize_device()
    elapsed_ms = (perf_counter() - start) * 1000.0
    return elapsed_ms / repeats


if RUN_INFERENCE_BENCHMARK:
    inputs, _ = next(iter(test_loader))
    inputs = inputs.to(
        device,
        non_blocking=(device.type == "cuda"),
    )

    benchmark_models = {
        "baseline": baseline,
        "hosvd_tucker2": model_h,
        "self_hooi_tucker2": model_o,
        "tensorly_partial_tucker": model_tl,
    }

    benchmark_rows = []

    for method, model in benchmark_models.items():
        ms_per_batch = benchmark_inference(
            model,
            inputs,
            warmup=BENCHMARK_WARMUP,
            repeats=BENCHMARK_REPEATS,
        )
        benchmark_rows.append({
            "method": method,
            "device": str(device),
            "batch_size": len(inputs),
            "warmup": BENCHMARK_WARMUP,
            "repeats": BENCHMARK_REPEATS,
            "ms_per_batch": ms_per_batch,
            "ms_per_image": ms_per_batch / len(inputs),
        })

    inference_benchmark = pd.DataFrame(benchmark_rows)
    display(inference_benchmark)

    benchmark_path = results_dir / "inference_benchmark.csv"
    inference_benchmark.to_csv(benchmark_path, index=False)
    print("saved:", benchmark_path)
else:
    print(
        "Inference benchmarkはskip。"
        "必要なら RUN_INFERENCE_BENCHMARK = True に変更してください。"
    )


## 完了条件

- [ ] `00_fundamentals/02_hooi.ipynb` でHOOIの更新原理を自分で実装する
- [ ] 04が02のbalanced rankを自動取得する
- [ ] 04の `tucker2_hooi_sweep()` を自分で実装する
- [ ] 自作HOSVD / 自作HOOI / TensorLyを同じrankで比較する
- [ ] weight relative errorとiterationごとの誤差を確認する
- [ ] decomposition timeを比較する
- [ ] 圧縮直後のvalidation loss / accuracy / accuracy dropを比較する
- [ ] 3手法でparameters / MACsが同じことを確認する
- [ ] 03 Fine-tuning結果は分解法比較と混ぜず、参考値として確認する
- [ ] 必要なら実測inference benchmarkを行う

ここまで確認できれば、**Tucker編はいったん終了**としてよい。
